# W&B Weave Hands-on: Trace a Simple Agent

짧은 시간 안에 **함수를 trace하는 법**, **Weave UI에서 trace tree를 읽는 법**, **간단한 tool-calling agent를 디버깅하는 법**을 익힙니다.

| 항목 | 내용 |
|---|---|
| 예상 시간 | Part 1: 10분, Part 2: 10–15분 |
| 필수 | W&B account, `WANDB_API_KEY` |
| 선택 | Part 2용 `OPENAI_API_KEY` |
| 실행 셀 | Part마다 1개, 총 2개 |
| 결과 | 기본 nested trace 1개, 선택적 agent trace 1개 |

## 진행 순서

1. **Part 1**: `@weave.op`로 기본 trace를 만들고 UI에서 직접 확인합니다.
2. **Part 2**: 같은 패턴을 간단한 function-calling agent에 적용합니다.
3. **선택**: Codex 세션 자체를 Weave로 보내는 integration을 설정합니다.

Part 1에는 외부 model 호출이 없으므로 OpenAI key나 API 비용 없이 실행할 수 있습니다.

---

## Part 1 · `@weave.op`로 첫 Trace 만들기

Weave tracing을 이해할 때는 아래 세 단어만 먼저 구분하면 됩니다.

| 개념 | 의미 | 이 실습에서 보이는 예 |
|---|---|---|
| **Op** | `@weave.op`를 붙인 추적 가능한 함수 | `research_helper`, `retrieve_notes` |
| **Call** | Op가 실제로 한 번 실행된 기록 | 입력, 출력, latency, error가 담긴 한 줄 |
| **Trace** | 한 작업 안에서 연결된 Call의 전체 트리 | root `research_helper`와 세 child Call |

Op 안에서 다른 Op를 호출하면 별도 설정 없이 parent-child 관계가 만들어집니다. 따라서 root에서는 전체 작업을 보고, child에서는 어느 단계의 입력이나 출력이 잘못됐는지 좁혀갈 수 있습니다.

### 실행 전

- `WANDB_API_KEY`를 환경변수로 설정하거나 실행 중 나타나는 `wandb.login()` 프롬프트를 사용합니다.
- 팀 project를 지정하려면 `WANDB_ENTITY`와 `WANDB_PROJECT`를 환경변수로 설정합니다.
- Part 2를 실행하려면 `OPENAI_API_KEY`가 필요하며, model 변경은 `OPENAI_MODEL`로 지정합니다.
- Dedicated Cloud만 `WANDB_BASE_URL`을 설정합니다. SaaS에서는 비워 둡니다.
- API key를 notebook 코드에 직접 적지 마세요.

아래 셀은 설치 → 로그인 → Weave 초기화 → nested trace 생성을 한 번에 수행합니다.

In [ ]:
# 0. Part 1과 Part 2에 필요한 SDK를 notebook 환경에 설치합니다.
%pip install -qU weave wandb openai

# 1. credential과 project 설정은 일반 환경변수에서 읽습니다.
import os

import wandb
import weave

# 2. W&B에 로그인하고 trace를 저장할 Weave project를 초기화합니다.
api_key = os.environ.get("WANDB_API_KEY")
if api_key:
    wandb.login(key=api_key)
else:
    wandb.login()

entity = os.environ.get("WANDB_ENTITY")
project = os.environ.get("WANDB_PROJECT", "weave-tracing-handson")
project_ref = f"{entity}/{project}" if entity else project

weave_client = weave.init(project_ref)
ENTITY = weave_client.entity
PROJECT = weave_client.project
WEAVE_URL = f"https://wandb.ai/{ENTITY}/{PROJECT}/weave/traces"

print("Weave project:", f"{ENTITY}/{PROJECT}")
print("Traces:", WEAVE_URL)

# 3. 실제 application의 단계를 작은 Op로 나눕니다.
# Op의 인자와 반환값은 각 Call의 input과 output으로 자동 기록됩니다.
@weave.op()
def normalize_question(question: str) -> str:
    return " ".join(question.strip().lower().split())


@weave.op()
def retrieve_notes(question: str) -> list[dict]:
    notes = [
        {
            "keyword": "trace",
            "note": "Trace tree에서 실패한 model 또는 tool Call을 먼저 찾습니다.",
        },
        {
            "keyword": "리뷰",
            "note": "리뷰 agent는 근거, 누락, 최종 판단 단계를 분리해 관찰합니다.",
        },
    ]
    matched = [item for item in notes if item["keyword"] in question]
    return matched or notes[:1]


@weave.op()
def draft_answer(question: str, notes: list[dict]) -> str:
    evidence = " ".join(item["note"] for item in notes)
    return f"질문: {question}\n답변: {evidence}"


# 4. root Op가 child Op들을 호출하면 하나의 nested trace가 만들어집니다.
@weave.op()
def research_helper(raw_question: str) -> str:
    question = normalize_question(raw_question)
    notes = retrieve_notes(question)
    return draft_answer(question, notes)


# 5. root Op를 한 번 실행해 실제 trace를 전송합니다.
answer = research_helper("리뷰 Agent의 Trace에서는 무엇을 확인해야 하나요?")
print("\nResult:\n", answer)
print("\nOpen the latest research_helper trace:\n", WEAVE_URL)

### Trace를 실제로 확인하기

셀 출력의 **Traces URL**을 열고 가장 최근 `research_helper`를 선택합니다. Dedicated Cloud에서는 `weave.init()`이 출력한 project link를 사용하세요.

1. **Call tree**: `research_helper` 아래에 `normalize_question` → `retrieve_notes` → `draft_answer`가 child Call로 보이는지 확인합니다.
2. **Inputs**: `retrieve_notes`를 눌러 정규화된 질문이 전달됐는지 봅니다.
3. **Outputs**: 검색된 note가 `draft_answer`의 입력으로 이어지는지 비교합니다.
4. **Timing**: 각 Call의 latency와 전체 trace duration을 확인합니다.
5. **Debugging 관점**: 답이 이상하다면 root 출력만 보지 말고, 처음 잘못된 값이 생긴 child Call을 찾습니다.

> **핵심**: 모든 helper를 trace할 필요는 없습니다. 입력·출력을 확인했을 때 디버깅 의사결정에 도움이 되는 단계만 Op로 만드세요.

시간이 부족하거나 OpenAI key가 없다면 여기까지가 필수 실습입니다.

---

## Part 2 · Simple Tool-calling Agent Trace

Part 1의 함수 트리를 agent에 그대로 적용합니다.

- `run_research_agent`: 사용자 요청부터 최종 답변까지 감싸는 **root Op**
- OpenAI call: Weave integration이 자동으로 기록하는 **model Call**
- `lookup_experiment`: agent가 선택해 실행하는 **tool Op**

이 셀은 `OPENAI_API_KEY`가 필요하며 소량의 API 비용이 발생합니다. 기본 model은 `gpt-4.1-mini`이고, `OPENAI_MODEL` 환경변수로 바꿀 수 있습니다. 첫 model Call에는 tool 사용을 요구해 실습할 때 tool trace가 확실히 생성되도록 했습니다.

In [ ]:
# 0. OpenAI key를 확인하고 client를 만듭니다.
import json

from openai import OpenAI

if not os.environ.get("OPENAI_API_KEY"):
    raise RuntimeError("Part 2를 실행하려면 OPENAI_API_KEY를 설정하세요.")

openai_client = OpenAI()
MODEL = os.environ.get("OPENAI_MODEL", "gpt-4.1-mini")

# 1. agent가 사용할 작은 tool을 Op로 만듭니다.
# 실제 autoresearch agent라면 이 자리에 run 조회, 논문 검색, metric 비교 등이 들어갑니다.
@weave.op()
def lookup_experiment(run_name: str) -> dict:
    experiments = {
        "baseline-a": {
            "status": "finished",
            "eval_accuracy": 0.71,
            "next_action": "candidate-b와 같은 seed로 비교",
        },
        "candidate-b": {
            "status": "finished",
            "eval_accuracy": 0.76,
            "next_action": "다른 seed 두 개로 재현성 확인",
        },
    }
    return experiments.get(
        run_name.lower(),
        {"status": "not_found", "next_action": "run 이름 확인"},
    )


# 2. model에게 tool의 이름, 설명, 입력 schema를 알려줍니다.
TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "lookup_experiment",
            "description": "Look up the status and metrics of an experiment run.",
            "parameters": {
                "type": "object",
                "properties": {
                    "run_name": {
                        "type": "string",
                        "description": "Experiment run name, such as candidate-b",
                    }
                },
                "required": ["run_name"],
                "additionalProperties": False,
            },
        },
    }
]
TOOL_MAP = {"lookup_experiment": lookup_experiment}


# 3. agent loop 전체를 root Op로 감쌉니다.
# weave.init 이후의 OpenAI SDK 호출은 자동 tracing되고, tool Op는 그 아래에 연결됩니다.
@weave.op()
def run_research_agent(user_message: str, max_turns: int = 3) -> str:
    messages = [
        {
            "role": "system",
            "content": (
                "You are a research experiment assistant. "
                "Use the provided tool and answer briefly in Korean."
            ),
        },
        {"role": "user", "content": user_message},
    ]

    for turn in range(max_turns):
        response = openai_client.chat.completions.create(
            model=MODEL,
            messages=messages,
            tools=TOOLS,
            tool_choice="required" if turn == 0 else "auto",
        )
        assistant_message = response.choices[0].message
        messages.append(assistant_message.model_dump(exclude_none=True))

        if not assistant_message.tool_calls:
            return assistant_message.content or "답변이 비어 있습니다."

        for tool_call in assistant_message.tool_calls:
            tool_name = tool_call.function.name
            tool_args = json.loads(tool_call.function.arguments)
            tool_result = TOOL_MAP[tool_name](**tool_args)
            messages.append(
                {
                    "role": "tool",
                    "tool_call_id": tool_call.id,
                    "content": json.dumps(tool_result, ensure_ascii=False),
                }
            )

    return "최대 turn 수에 도달했습니다."


# 4. 한 질문만 실행해 agent trace 하나를 만듭니다.
agent_answer = run_research_agent(
    "candidate-b 실험 상태를 확인하고 다음 액션을 알려줘."
)
print("Agent answer:", agent_answer)
print("\nOpen the latest run_research_agent trace:\n", WEAVE_URL)

### Agent Trace를 읽는 순서

가장 최근 `run_research_agent` trace를 열고 다음 순서로 내려갑니다.

1. **Root**: user message와 최종 answer가 맞는지 확인합니다.
2. **첫 model Call**: model이 `lookup_experiment`를 선택했고 argument로 `candidate-b`를 만들었는지 확인합니다.
3. **Tool Call**: tool output의 status, accuracy, next action이 올바른지 확인합니다.
4. **마지막 model Call**: tool 결과가 최종 한국어 답변에 반영됐는지 확인합니다.
5. **운영 정보**: model Call의 latency, token usage, cost와 error 여부를 확인합니다.

문제가 생겼을 때 해석하는 방법도 단순합니다.

| 증상 | 먼저 볼 Call |
|---|---|
| 엉뚱한 tool을 선택함 | 첫 model Call의 input과 tool schema |
| tool argument가 잘못됨 | 첫 model Call의 output |
| 데이터가 틀림 | `lookup_experiment` input/output |
| 데이터는 맞지만 답변이 틀림 | 마지막 model Call |

## 완료 체크리스트

- [ ] `@weave.op`가 함수의 input, output, code, timing을 기록한다는 것을 이해했다.
- [ ] 하나의 Trace가 parent-child Call tree라는 것을 이해했다.
- [ ] `research_helper` trace를 Weave UI에서 직접 열었다.
- [ ] 선택적으로 `run_research_agent`의 model/tool Call을 확인했다.
- [ ] 자신의 agent에서 먼저 trace할 의미 있는 단계 3–5개를 정할 수 있다.

---

## 선택 · Codex 세션을 Weave로 Trace하기

Notebook 코드가 아니라 **Codex를 실행하는 로컬 터미널**에서 설정합니다. 이 integration은 Codex의 turn, model call, reasoning step, tool execution을 자동으로 Weave에 보냅니다.

기본 설정은 prompt, response, reasoning, tool argument, command output과 file content를 캡처하며 자동 민감정보 제거를 제공하지 않습니다. 내용 전송이 허용되는지 먼저 확인하세요. 구조, token usage, model, timing만 보내려면 Codex 실행 전에 `WEAVE_CODEX_CAPTURE_CONTENT=0`을 설정합니다.

```bash
npm install -g weave-codex
wandb login
export WEAVE_PROJECT="YOUR-TEAM/YOUR-PROJECT"

# 내용 없이 구조와 사용량만 전송하려면 다음 줄도 설정합니다.
export WEAVE_CODEX_CAPTURE_CONTENT=0

weave-codex install
weave-codex status
```

다음 Codex 실행 때 hook을 승인합니다. 이후 Weave의 **Agents**에서는 대화 단위로, **Traces**에서는 raw span tree로 확인할 수 있습니다. 보안 정책상 content 전송이 불가능하면 integration을 설치하지 마세요.

## 참고 자료

- [Understand Ops, Calls, and Traces](https://docs.wandb.ai/weave/guides/tracking/tracing)
- [Trace your code](https://docs.wandb.ai/weave/guides/tracking/create-call)
- [OpenAI integration](https://docs.wandb.ai/weave/guides/integrations/openai)
- [Codex plugin](https://docs.wandb.ai/weave/guides/integrations/agents/codex-harness)
- 원본: [hw-oh/weave-initial-course](https://github.com/hw-oh/weave-initial-course/tree/a6a2b15872ffe4239c531eda33f3a55afea6c5b5/notebooks)

이 notebook은 원본 코스의 setup, traces and ops, simple agent 내용을 해커톤용 짧은 tracing 실습으로 재구성했습니다.